# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools



## Assignment Pipeline

1. Logging Setup 
2. Tools
3. Agent Logic
4. Test Cases
5. Interactive Mode
6. Results
7. Limitations

## 1. Logging Setup

In [35]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger("agent_logger")

## 2. Tools

In [28]:
# 🛠️ TOOL 1: Calculator
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    return str(eval(expression))

In [29]:
# 🛠️ TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

In [123]:
# 🛠️ TOOL 3: Unit Converter

def convert_units(text: str) -> str:
    """Convert a value between common units (km/meters,km/miles, kg/lbs, celsius/fahrenheit)."""
    try:
        words = text.split()
        value = float(words[0])
        unit_from = words[1].lower()
        unit_to = words[-1].lower()

        conversions = {
            ("km", "meters"): lambda v: v * 1000,
            ("km", "meter"): lambda v: v * 1000,
            ("meters", "km"): lambda v: v / 1000,
            ("meter", "km"): lambda v: v / 1000,
            ("km", "miles"): lambda v: v * 0.621371,
            ("km", "mile"): lambda v: v * 0.621371,
            ("miles", "km"): lambda v: v * 1.60934,
            ("mile", "km"): lambda v: v * 1.60934,
            ("kg", "lbs"): lambda v: v * 2.20462,
            ("lbs", "kg"): lambda v: v * 0.453592,
            ("celsius", "fahrenheit"): lambda v: (v * 9/5) + 32,
            ("fahrenheit", "celsius"): lambda v: (v - 32) * 5/9,
            ("meters", "miles"): lambda v: v * 0.000621371,
            ("meter", "mile"): lambda v: v * 0.000621371,
            ("miles", "meters"): lambda v: v * 1609.34,
            ("mile", "meter"): lambda v: v * 1609.34,
        }

        key = (unit_from, unit_to)
        if key in conversions:
            result = conversions[key](value)
            return f"{value} {unit_from} = {round(result, 2)} {unit_to}"
        else:
            return "Unsupported unit conversion"
    except Exception:
        return "Could not perform conversion request"

In [94]:
# 🛠️ TOOL 4: Text Summarizer

import re
from collections import Counter

def summarize_text(text: str, num_sentences: int = 2) -> str:
    """Summarizer: picks the most important sentence(s) based on word frequency."""
    try:
        sentences = re.split(r'(?<=[.!?]) +', text.strip())
        if len(sentences) <= num_sentences:
            return text.strip()

        words = re.findall(r'\w+', text.lower())
        stopwords = {"the", "is", "a", "an", "and", "of", "to", "in", "on", "for", "it", "this", "that", "with", "as", "are", "was", "were"}
        word_freq = Counter(w for w in words if w not in stopwords)

        sentence_scores = {}
        for sentence in sentences:
            sentence_words = re.findall(r'\w+', sentence.lower())
            score = sum(word_freq.get(w, 0) for w in sentence_words)
            sentence_scores[sentence] = score

        top_sentences = sorted(sentence_scores, key=sentence_scores.get, reverse=True)[:num_sentences]
        top_sentences_in_order = [s for s in sentences if s in top_sentences]
        return " ".join(top_sentences_in_order)
    except Exception:
        return "Could not summarize text"

In [100]:
# 🛠️ TOOL 5: Currency Converter

def convert_currency(text: str) -> str:
    """Convert a value between common currencies using fixed rates (not live)."""
    try:
        words = text.split()
        value = float(words[0])
        currency_from = words[1].upper()
        currency_to = words[-1].upper()

        # Note: rates are illustrative/fixed, not live market rates
        rates_to_usd = {
            "USD": 1.0,
            "INR": 0.012,
            "EUR": 1.09,
            "GBP": 1.27,
            "JPY": 0.0064,
        }

        if currency_from not in rates_to_usd or currency_to not in rates_to_usd:
            return "Unsupported currency"

        value_in_usd = value * rates_to_usd[currency_from]
        result = value_in_usd / rates_to_usd[currency_to]
        return f"{value} {currency_from} = {round(result, 2)} {currency_to}"
    except Exception:
        return "Could not perform currency conversion"

## 3. Agent Logic

Routes each incoming query to the correct tool based on trigger words, then returns a structured JSON-style response: `{"type": ..., "result": ...}`.

In [129]:
# 🤖 AGENT FUNCTION (TO IMPLEMENT)

def agent(query: str):
    query_lower = query.lower()
    logger.info(f"Received query: {query}")

    calculate_triggers = ["calculate", "calc", "compute", "solve"]
    keyword_triggers = ["keywords", "keyword", "imp words", "important word"]
    summarize_triggers = ["summarize", "summary", "summarise"]
    currency_codes = ["usd", "inr", "eur", "gbp", "jpy"]

    if any(word in query_lower for word in calculate_triggers):
        expression = query_lower
        for word in calculate_triggers:
            expression = expression.replace(word, "")
        expression = expression.strip()
        try:
            result = calculator(expression)
            logger.info(f"Routed to calculator | expression='{expression}' | result={result}")
            return {
                "type": "calculation",
                "result": result
            }
        except Exception:
            logger.error(f"Calculation failed | expression='{expression}'")
            return {
                "type": "error",
                "result": "Could not evaluate the expression"
            }

    elif any(word in query_lower for word in keyword_triggers):
        text = query_lower
        for word in keyword_triggers:
            text = text.replace(f"extract {word} from", "")
        text = text.strip()
        result = extract_keywords(text)
        logger.info(f"Routed to keyword extractor | text='{text}' | result={result}")
        return {
            "type": "keywords",
            "result": result
        }

    elif any(word in query_lower for word in summarize_triggers):
        text_to_summarize = query_lower
        for word in summarize_triggers:
            text_to_summarize = text_to_summarize.replace(word, "")
        text_to_summarize = text_to_summarize.replace(":", "").strip()
        result = summarize_text(text_to_summarize)
        logger.info(f"Routed to summarizer | text='{text_to_summarize[:50]}...' | result={result[:50]}...")
        return {
            "type": "summary",
            "result": result
        }

    elif any(code in query_lower for code in currency_codes) and "to" in query_lower:
        text = query_lower.replace("convert", "").strip()
        result = convert_currency(text)
        logger.info(f"Routed to currency converter | text='{text}' | result={result}")
        return {
            "type": "currency",
            "result": result
        }

    elif "convert" in query_lower and "to" in query_lower:
        text = query_lower.replace("convert", "").strip()
        result = convert_units(text)
        logger.info(f"Routed to unit converter | text='{text}' | result={result}")
        return {
            "type": "conversion",
            "result": result
        }

    logger.info(f"Routed to general response | query='{query}'")
    return {
        "type": "general",
        "result": f"This is a general query: {query}"
    }

## 4. Test Cases

In [116]:
# ✏️ Test Cases

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?"
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

2026-06-29 01:06:49,560 - INFO - Received query: Calculate 20 + 5
2026-06-29 01:06:49,561 - INFO - Routed to calculator | expression='20 + 5' | result=25
2026-06-29 01:06:49,562 - INFO - Received query: Extract keywords from Artificial Intelligence is transforming industries
2026-06-29 01:06:49,563 - INFO - Routed to keyword extractor | text='artificial intelligence is transforming industries' | result=['transforming', 'intelligence', 'artificial', 'industries']
2026-06-29 01:06:49,564 - INFO - Received query: What is machine learning?
2026-06-29 01:06:49,564 - INFO - Routed to general response | query='What is machine learning?'


Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['transforming', 'intelligence', 'artificial', 'industries']}
--------------------------------------------------
Query: What is machine learning?
Response: {'type': 'general', 'result': 'This is a general query: What is machine learning?'}
--------------------------------------------------


In [117]:
print(agent("Calculate banana + 5"))

2026-06-29 01:06:50,455 - INFO - Received query: Calculate banana + 5
2026-06-29 01:06:50,457 - ERROR - Calculation failed | expression='banana + 5'


{'type': 'error', 'result': 'Could not evaluate the expression'}


In [118]:
print(agent("Calculate 20 + 5"))

2026-06-29 01:06:53,173 - INFO - Received query: Calculate 20 + 5
2026-06-29 01:06:53,174 - INFO - Routed to calculator | expression='20 + 5' | result=25


{'type': 'calculation', 'result': '25'}


In [119]:
print(agent("convert 10 km to meter"))
print(agent("convert 10 km to meters"))
print(agent("convert 100 fahrenheit to celsius"))
print(agent("calculate 20 + 5"))
print(agent("extract keywords from this is a simple test"))
print(agent("what is the weather today"))

2026-06-29 01:06:54,721 - INFO - Received query: convert 10 km to meter
2026-06-29 01:06:54,722 - INFO - Routed to unit converter | text='10 km to meter' | result=10.0 km = 10000.0 meter
2026-06-29 01:06:54,723 - INFO - Received query: convert 10 km to meters
2026-06-29 01:06:54,723 - INFO - Routed to unit converter | text='10 km to meters' | result=10.0 km = 10000.0 meters
2026-06-29 01:06:54,724 - INFO - Received query: convert 100 fahrenheit to celsius
2026-06-29 01:06:54,725 - INFO - Routed to unit converter | text='100 fahrenheit to celsius' | result=100.0 fahrenheit = 37.78 celsius
2026-06-29 01:06:54,726 - INFO - Received query: calculate 20 + 5
2026-06-29 01:06:54,727 - INFO - Routed to calculator | expression='20 + 5' | result=25
2026-06-29 01:06:54,727 - INFO - Received query: extract keywords from this is a simple test
2026-06-29 01:06:54,728 - INFO - Routed to keyword extractor | text='this is a simple test' | result=['simple']
2026-06-29 01:06:54,728 - INFO - Received quer

{'type': 'conversion', 'result': '10.0 km = 10000.0 meter'}
{'type': 'conversion', 'result': '10.0 km = 10000.0 meters'}
{'type': 'conversion', 'result': '100.0 fahrenheit = 37.78 celsius'}
{'type': 'calculation', 'result': '25'}
{'type': 'keywords', 'result': ['simple']}
{'type': 'general', 'result': 'This is a general query: what is the weather today'}


In [108]:
text = "Artificial Intelligence is transforming industries worldwide. Many companies are adopting AI to improve efficiency. AI helps automate repetitive tasks and reduce costs. However, ethical concerns around AI remain important to address."
print(summarize_text(text))

AI helps automate repetitive tasks and reduce costs. However, ethical concerns around AI remain important to address.


In [110]:
print(convert_currency("100 USD to INR"))
print(convert_currency("50 EUR to GBP"))

100.0 USD = 8333.33 INR
50.0 EUR = 42.91 GBP


In [111]:
print(agent("100 usd to inr"))
print(agent("convert 10 km to miles"))
print(agent("summarize this: AI is transforming industries. Many companies adopt AI. AI reduces costs. Ethics matter."))
print(agent("calculate 20 + 5"))

2026-06-29 00:58:30,407 - INFO - Received query: 100 usd to inr
2026-06-29 00:58:30,409 - INFO - Routed to currency converter | text='100 usd to inr' | result=100.0 USD = 8333.33 INR
2026-06-29 00:58:30,410 - INFO - Received query: convert 10 km to miles
2026-06-29 00:58:30,410 - INFO - Routed to unit converter | text='10 km to miles' | result=10.0 km = 6.21 miles
2026-06-29 00:58:30,411 - INFO - Received query: summarize this: AI is transforming industries. Many companies adopt AI. AI reduces costs. Ethics matter.
2026-06-29 00:58:30,411 - INFO - Routed to summarizer | text='this ai is transforming industries. many companies...' | result=this ai is transforming industries. many companies...
2026-06-29 00:58:30,412 - INFO - Received query: calculate 20 + 5
2026-06-29 00:58:30,412 - INFO - Routed to calculator | expression='20 + 5' | result=25


{'type': 'currency', 'result': '100.0 USD = 8333.33 INR'}
{'type': 'conversion', 'result': '10.0 km = 6.21 miles'}
{'type': 'summary', 'result': 'this ai is transforming industries. many companies adopt ai.'}
{'type': 'calculation', 'result': '25'}


## 5. Interactive Mode

In [131]:
# 🎯 Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))

Enter query (type 'exit' to stop):  solve (8*(9+17))//2


2026-06-29 01:25:49,127 - INFO - Received query: solve (8*(9+17))//2
2026-06-29 01:25:49,133 - INFO - Routed to calculator | expression='(8*(9+17))//2' | result=104


Response: {'type': 'calculation', 'result': '104'}


Enter query (type 'exit' to stop):  what are the imp words in learning agentic ai is crucial 


2026-06-29 01:26:08,705 - INFO - Received query: what are the imp words in learning agentic ai is crucial 
2026-06-29 01:26:08,711 - INFO - Routed to keyword extractor | text='what are the imp words in learning agentic ai is crucial' | result=['learning', 'crucial', 'agentic', 'words']


...

Response: {'type': 'summary', 'result': 'can you  this severe flooding swept through coastal regions on monday after days of heavy rainfall. officials warn that water levels could rise further over the next 48 hours.'}


Enter query (type 'exit' to stop):  convert 100 km to meters 


2026-06-29 01:28:05,474 - INFO - Received query: convert 100 km to meters 
2026-06-29 01:28:05,479 - INFO - Routed to unit converter | text='100 km to meters' | result=100.0 km = 100000.0 meters


Response: {'type': 'conversion', 'result': '100.0 km = 100000.0 meters'}


Enter query (type 'exit' to stop):  exit


## 6. Results

- All 3 required tools worked correctly and gave back the right JSON format.
- Tested error handling by passing invalid stuff like division by zero which it correctly returned an "error" type instead of just breaking or giving a wrong answer.
- Added 3 extra tools named unit converter, currency converter and text summarizer ; tested all of them through the agent rather than them being on their own.
- Used Python's logging module so I could see exactly which query came in and which tool it got routed to.

## 7. Limitations

- Since the way queries get sent to a tool is based on matching exact words, adding a generic words can cause it to match the wrong tool.
- The unit and currency converter only work if you type the unit exactly right which is written as "meter" or "meters" so it had to be added as two separate entries otherwise one of them just wouldn't match and the tool would fail into a general query format
- The summarizer doesn't actually understand the text it works on frequency of the words